# Temporal Fusion Transformer — Walmart Store Sales Forecasting

Model experiment notebook for the TFT branch of the multi-model comparison
(LightGBM, XGBoost, DLinear, TFT). The Temporal Fusion Transformer (Lim et
al., 2021) combines an LSTM encoder-decoder, gated variable-selection
networks per input, and interpretable multi-head attention — built on
`pytorch_forecasting`'s `TemporalFusionTransformer` rather than hand-rolled.
Unlike DLinear (~150 lines, easy to validate correctness of by hand), TFT
has enough moving parts that reimplementing it from scratch without a GPU to
iterate against would be high-risk for low payoff — `utils/tft.py` wraps
the library's `TimeSeriesDataSet`/`TemporalFusionTransformer` with this
project's data-prep and raw-input-inference conventions instead.

**Requires GPU** (Colab), unlike DLinear. DLinear was ~2,400 parameters —
two linear layers, nothing for a GPU to meaningfully parallelize. TFT is a
genuinely heavy architecture (LSTM recurrence, multi-head attention scaling
quadratically with sequence length, gated residual networks per input
variable) that easily reaches the hundreds of thousands of parameters even
at modest settings — 2-3 orders of magnitude more than DLinear, and exactly
the regime where GPU parallelism pays off. CPU training of a comparably
sized workload would plausibly take many hours to days.

**Cells written but not executed** — meant to run on Colab (GPU). Every
piece of logic (`build_tft_panel`, `TimeSeriesDataSet` construction, the
multi-split CV/WMAE-evaluation orchestration, the raw-input
`TFTForecastPipeline`, including partial-request row alignment) was
independently validated against `utils/tft.py` with a throwaway Python 3.11
CPU environment and synthetic data — tiny scale, few epochs, just enough to
confirm the API is wired correctly, not to judge real performance.

**How this notebook differs from the other three, and why:**

- **Uses `features.csv`/`stores.csv`, unlike DLinear.** DLinear stayed
  deliberately minimal (history + `IsHoliday` only) because it's just two
  linear layers with no mechanism to figure out which extra inputs matter.
  TFT's variable-selection network exists specifically to learn that —
  feeding it only 2 inputs would waste the architecture. `features.csv`
  covers the entire competition date range (through 2013-07-26, past even
  Kaggle's `test.csv`), so `Temperature`/`Fuel_Price`/`CPI`/`Unemployment`/
  `MarkDown1-5` are legitimately "known" covariates for any date here, not
  just observed history — same treatment the tree notebooks gave them. No
  manual feature-selection step (tree notebooks' Step 4) either: TFT's
  variable selection network does that job internally, including for the
  markdowns the tree models mostly found useless — better to let TFT make
  that call itself than pre-filter with a different model's judgment.
- **Direct 52-week horizon for the real forecasts, not recursive
  rollout.** DLinear had to chain 13-week blocks recursively to cover the
  52-week holdout, feeding predictions back in as pseudo-history — a real
  source of compounding error. TFT is architecturally built for direct
  multi-horizon output, so the holdout-eval and final production models use
  `max_prediction_length=52` directly (covering Kaggle's real 39-week
  `test.csv` too, as a strict subset) — no recursion, no compounding error.
  CV/tuning still uses `max_prediction_length=13`, for the same reason
  DLinear did: `local_train_raw` is only 91 weeks, and `max_encoder_length
  (52) + max_prediction_length` must fit inside a split's training range.
- **Three validation splits**, same `[65, 72, 78]`-week boundaries as
  DLinear's — the same 91-week budget constrains both models identically,
  so there's no reason to redesign this from scratch.
- **Quantile loss, not a single point estimate.** `QuantileLoss` (TFT's
  standard choice) gives calibrated uncertainty bands, not just a WMAE
  number — genuinely something none of the other three models here provide.
  The median (0.5 quantile) is used as the point forecast for WMAE, so
  results stay comparable to LightGBM/XGBoost/DLinear.
- **Learning rate found once via Lightning's LR finder**, not grid-searched
  — standard practice for this architecture, and cheaper than adding it as
  a third grid dimension. `hidden_size`/`attention_head_size` are still
  grid-searched (both change real model capacity in ways an LR finder can't
  substitute for).
- **Richer interpretability plots**, using `pytorch_forecasting`'s native
  `plot_interpretation`/`plot_prediction` rather than hand-rolled — TFT's
  built-in variable-importance and attention outputs are genuine, principled
  quantities (not DLinear's crude linear-weight-magnitude proxy), and are
  worth showing in full.

## Table of Contents
1. [Setup](#1)
2. [Local train/test split](#2)
3. [Sequence construction & validation splits](#3)
4. [Model architecture](#4)
5. [Hyperparameter tuning](#5)
6. [MLflow logging](#6)
7. [Plots](#7)
8. [Full pipeline](#8)

In [1]:
!git clone https://github.com/tamari1990/ml-final-project.git
%cd ml-final-project
!git pull

fatal: destination path 'ml-final-project' already exists and is not an empty directory.
/content/ml-final-project
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 850 bytes | 850.00 KiB/s, done.
From https://github.com/tamari1990/ml-final-project
   ab51eae..7522694  main       -> origin/main
Updating ab51eae..7522694
Fast-forward
 utils/tft.py | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)


In [2]:
!pip install -q pytorch-forecasting lightning

<a id='1'></a>
## 1. Setup

In [3]:
import warnings
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping
from lightning.pytorch.tuner import Tuner

from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.metrics import QuantileLoss

warnings.filterwarnings('ignore', category=DeprecationWarning)

from utils.tft import build_tft_panel, make_training_dataset, TFTForecastPipeline
from utils.feature_engineering import HOLIDAY_DATES
from utils.metrics import wmae

pd.set_option('display.max_columns', 50)

SEED = 42
pl.seed_everything(SEED)

ACCELERATOR = 'gpu' if torch.cuda.is_available() else 'cpu'
print('accelerator:', ACCELERATOR)
if ACCELERATOR == 'gpu':
    print('GPU:', torch.cuda.get_device_name(0))

DATA_DIR = 'data/raw/walmart-recruiting-store-sales-forecasting/'

train = pd.read_csv(DATA_DIR + 'train.csv', parse_dates=['Date'])
test = pd.read_csv(DATA_DIR + 'test.csv', parse_dates=['Date'])
features = pd.read_csv(DATA_DIR + 'features.csv', parse_dates=['Date'])
stores = pd.read_csv(DATA_DIR + 'stores.csv')

train = train.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

print(f'train : {train.shape}, {train.Date.min().date()} -> {train.Date.max().date()}, '
      f'{train.Date.nunique()} weeks, {train[["Store","Dept"]].drop_duplicates().shape[0]} series')
print(f'test  : {test.shape}, {test.Date.min().date()} -> {test.Date.max().date()}, {test.Date.nunique()} weeks')
print(f'features.csv: {features.Date.min().date()} -> {features.Date.max().date()} (covers test.csv too)')
print(f'stores.csv: {stores.shape[0]} stores, types {stores.Type.value_counts().to_dict()}')

INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


accelerator: gpu
GPU: Tesla T4
train : (421570, 5), 2010-02-05 -> 2012-10-26, 143 weeks, 3331 series
test  : (115064, 4), 2012-11-02 -> 2013-07-26, 39 weeks
features.csv: 2010-02-05 -> 2013-07-26 (covers test.csv too)
stores.csv: 45 stores, types {'A': 22, 'B': 17, 'C': 6}


<a id='2'></a>
## 2. Local Train/Test Split

Identical split to every other notebook in this project (same reasoning:
last 52 weeks of `train.csv` held out locally, standing in for the
target-less Kaggle `test.csv` during development).

In [4]:
unique_dates = np.sort(train['Date'].unique())
cutoff_date = unique_dates[-52]

local_train_raw = train[train['Date'] < cutoff_date].copy()
local_test_raw = train[train['Date'] >= cutoff_date].copy()

print(f'cutoff date (first date held out): {pd.Timestamp(cutoff_date).date()}')
print(f'local_train_raw: {local_train_raw.shape}, {local_train_raw.Date.min().date()} -> {local_train_raw.Date.max().date()}  ({local_train_raw.Date.nunique()} weeks)')
print(f'local_test_raw : {local_test_raw.shape}, {local_test_raw.Date.min().date()} -> {local_test_raw.Date.max().date()}  ({local_test_raw.Date.nunique()} weeks)')

cutoff date (first date held out): 2011-11-04
local_train_raw: (267184, 5), 2010-02-05 -> 2011-10-28  (91 weeks)
local_test_raw : (154386, 5), 2011-11-04 -> 2012-10-26  (52 weeks)


In [5]:
def holidays_in_range(dates_series):
    dates_set = set(pd.to_datetime(dates_series))
    present = {}
    for name, dates in HOLIDAY_DATES.items():
        matched = [d for d in pd.to_datetime(dates) if d in dates_set]
        if matched:
            present[name] = [d.date() for d in matched]
    return present


print('local_train_raw holidays:', holidays_in_range(local_train_raw['Date']))
print('local_test_raw  holidays:', holidays_in_range(local_test_raw['Date']))
print()
print('Kaggle test.csv holidays (reference, no target):', holidays_in_range(test['Date']))

local_train_raw holidays: {'SuperBowl': [datetime.date(2010, 2, 12), datetime.date(2011, 2, 11)], 'LaborDay': [datetime.date(2010, 9, 10), datetime.date(2011, 9, 9)], 'Thanksgiving': [datetime.date(2010, 11, 26)], 'Christmas': [datetime.date(2010, 12, 31)]}
local_test_raw  holidays: {'SuperBowl': [datetime.date(2012, 2, 10)], 'LaborDay': [datetime.date(2012, 9, 7)], 'Thanksgiving': [datetime.date(2011, 11, 25)], 'Christmas': [datetime.date(2011, 12, 30)]}

Kaggle test.csv holidays (reference, no target): {'SuperBowl': [datetime.date(2013, 2, 8)], 'Thanksgiving': [datetime.date(2012, 11, 23)], 'Christmas': [datetime.date(2012, 12, 28)]}


<a id='3'></a>
## 3. Sequence Construction & Validation Splits

`LOOKBACK = 52` weeks of encoder context. Two different `HORIZON` values,
for two different reasons:

- `HORIZON_CV = 13`: for tuning only. A training window needs
  `LOOKBACK + HORIZON` consecutive weeks, and `local_train_raw` is only 91
  weeks — the same constraint DLinear hit. `CV_SPLIT_TRAIN_WEEKS = [65, 72,
  78]` is the same spread DLinear used, for the same reason (the max spread
  a 91-week budget with a 65-week minimum window allows).
- `HORIZON_FINAL = 52`: for the holdout-eval and final production models
  (Step 7/8), once there's no more CV budget constraint to respect. TFT
  predicts this directly in one shot — no recursive block-chaining needed,
  unlike DLinear.

`utils.tft.build_tft_panel` handles the data prep: merges `features.csv`/
`stores.csv`, reindexes every series onto the full weekly calendar (gaps ->
`Weekly_Sales=0`, same policy as `utils.dlinear.build_full_calendar_panel`),
fills the `MarkDown1-5` columns' ~50-64% missingness with 0 (the promotion
program didn't start until 2011-11-11 — "not running" is the natural
reading, not something to impute), and builds the `time_idx`/calendar/
`sample_weight` columns `TimeSeriesDataSet` needs. No series is excluded —
`utils.tft.make_training_dataset` builds a `TimeSeriesDataSet` over every
(Store, Dept) pair present in a split's training range.

In [6]:
LOOKBACK = 52
HORIZON_CV = 13
HORIZON_FINAL = 52
CV_SPLIT_TRAIN_WEEKS = [65, 72, 78]

local_train_dates = np.sort(local_train_raw['Date'].unique())


def build_cv_split(train_end):
    tr_dates = local_train_dates[:train_end]
    va_dates = local_train_dates[train_end:train_end + HORIZON_CV]
    assert len(va_dates) == HORIZON_CV, 'val window must be exactly one HORIZON_CV block for a direct (non-recursive) validation score'

    tr_df = local_train_raw[local_train_raw['Date'].isin(tr_dates)]
    va_df = local_train_raw[local_train_raw['Date'].isin(va_dates)]

    # panel spans train+val: val rows supply the true target/IsHoliday for
    # scoring, and the model never sees them during training since
    # make_training_dataset only keeps windows up to (panel max time_idx -
    # HORIZON_CV), i.e. strictly the tr_df portion.
    panel = build_tft_panel(pd.concat([tr_df, va_df], ignore_index=True), features, stores)
    training_ds = make_training_dataset(panel, LOOKBACK, HORIZON_CV)
    val_ds = TimeSeriesDataSet.from_dataset(training_ds, panel, predict=True, stop_randomization=True)

    return {'train_end': train_end, 'va_dates': va_dates, 'panel': panel, 'training_ds': training_ds, 'val_ds': val_ds}


cv_splits = [build_cv_split(te) for te in CV_SPLIT_TRAIN_WEEKS]

for s in cv_splits:
    n_series = s['panel'][['Store', 'Dept']].drop_duplicates().shape[0]
    print(f"split train_end={s['train_end']}: val {pd.Timestamp(s['va_dates'][0]).date()} -> {pd.Timestamp(s['va_dates'][-1]).date()} "
          f"holidays={list(holidays_in_range(s['va_dates']).keys())}")
    print(f"  {n_series} series, {len(s['training_ds'])} training samples, {len(s['val_ds'])} val series")

split train_end=65: val 2011-05-06 -> 2011-07-29 holidays=[]
  3241 series, 249557 training samples, 3241 val series
split train_end=72: val 2011-06-24 -> 2011-09-16 holidays=['LaborDay']
  3248 series, 272832 training samples, 3248 val series
split train_end=78: val 2011-08-05 -> 2011-10-28 holidays=['LaborDay']
  3254 series, 292860 training samples, 3254 val series


**WMAE evaluation** for a `TimeSeriesDataSet`-based split: predict
(median quantile) on `val_ds`, then join each prediction back to the
split's own `panel` by `(Store, Dept, time_idx)` to recover the true
`Weekly_Sales`/`IsHoliday` for that exact window — `result.index` (from
`model.predict(..., return_index=True)`) gives the starting `time_idx` per
series, safe across however many batches `predict()` internally uses.

In [7]:
def evaluate_wmae(model, split, horizon):
    result = model.predict(split['val_ds'], mode='prediction', return_index=True, batch_size=256, num_workers=0)
    preds = np.clip(result.output.cpu().numpy(), 0, None)
    index = result.index

    panel = split['panel']
    trues, holidays, preds_flat = [], [], []
    for row_i in range(preds.shape[0]):
        store, dept = str(int(index.iloc[row_i]['Store'])), str(int(index.iloc[row_i]['Dept']))
        start_idx = int(index.iloc[row_i]['time_idx'])
        sub = panel[(panel['Store'].astype(str) == store) & (panel['Dept'].astype(str) == dept) &
                    (panel['time_idx'] >= start_idx) & (panel['time_idx'] < start_idx + horizon)]
        sub = sub.sort_values('time_idx')
        if len(sub) != horizon:
            continue  # series without a full horizon of true data in this panel (shouldn't happen for CV splits, guards Step 7/8's shorter local_test_raw tail)
        trues.extend(sub['Weekly_Sales'].tolist())
        holidays.extend(sub['IsHoliday'].tolist())
        preds_flat.extend(preds[row_i, :horizon].tolist())
    return wmae(trues, preds_flat, holidays)

Baseline TFT (`hidden_size=16`, `attention_head_size=1`, a fixed
starter `learning_rate=0.03`) across all 3 splits, to confirm the training
harness works end-to-end and produce a sane mean WMAE before tuning (Step
5) builds on top of it — same purpose as every other notebook's baseline
step. `EarlyStopping` (Lightning's built-in callback, monitoring
`val_loss`) replaces DLinear's hand-written early-stopping loop.

In [8]:
def train_one_split(split, hidden_size, attention_head_size, learning_rate, max_epochs, patience,
                     accelerator=ACCELERATOR, log_mlflow_prefix=None):
    train_dl = split['training_ds'].to_dataloader(train=True, batch_size=128, num_workers=0)
    val_dl = split['val_ds'].to_dataloader(train=False, batch_size=128, num_workers=0)

    tft = TemporalFusionTransformer.from_dataset(
        split['training_ds'], learning_rate=learning_rate, hidden_size=hidden_size,
        attention_head_size=attention_head_size, dropout=0.1,
        hidden_continuous_size=max(hidden_size // 2, 4), loss=QuantileLoss(), optimizer='adam',
    )
    callbacks = [EarlyStopping(monitor='val_loss', patience=patience, mode='min')]
    if log_mlflow_prefix:
        from lightning.pytorch.callbacks import Callback

        class _MlflowStepLogger(Callback):
            def on_validation_epoch_end(self, trainer, pl_module):
                loss = trainer.callback_metrics.get('val_loss')
                if loss is not None:
                    mlflow.log_metric(f'{log_mlflow_prefix}_val_loss', float(loss), step=trainer.current_epoch)
        callbacks.append(_MlflowStepLogger())

    trainer = pl.Trainer(
        max_epochs=max_epochs, accelerator=accelerator, enable_progress_bar=False, enable_model_summary=False,
        logger=False, enable_checkpointing=False, callbacks=callbacks,
    )
    trainer.fit(tft, train_dataloaders=train_dl, val_dataloaders=val_dl)
    val_wmae = evaluate_wmae(tft, split, HORIZON_CV)
    return tft, val_wmae, trainer.current_epoch


def train_and_score_across_splits(cv_splits, hidden_size, attention_head_size, learning_rate,
                                   max_epochs, patience, log_mlflow=False):
    """Trains one fresh model per split (mirrors every other notebook's
    per-fold/per-split pattern), returns (mean_val_wmae, per_split_wmaes,
    per_split_epochs)."""
    split_wmaes, split_epochs = [], []
    for i, s in enumerate(cv_splits):
        prefix = f'split{i}' if log_mlflow else None
        _, val_wmae, epoch = train_one_split(
            s, hidden_size, attention_head_size, learning_rate, max_epochs, patience, log_mlflow_prefix=prefix,
        )
        split_wmaes.append(val_wmae)
        split_epochs.append(epoch)
    return float(np.mean(split_wmaes)), split_wmaes, split_epochs


BASELINE_PARAMS = dict(hidden_size=16, attention_head_size=1)
BASELINE_LR = 0.03
MAX_EPOCHS = 50
PATIENCE = 8

print('Training baseline TFT across all 3 splits...')
baseline_val_wmae, baseline_split_wmaes, baseline_split_epochs = train_and_score_across_splits(
    cv_splits, **BASELINE_PARAMS, learning_rate=BASELINE_LR, max_epochs=MAX_EPOCHS, patience=PATIENCE,
)
print(f'Baseline mean val WMAE: {baseline_val_wmae:.2f}  (per-split: {[round(w, 1) for w in baseline_split_wmaes]}, '
      f'epochs: {baseline_split_epochs})')

Training baseline TFT across all 3 splits...


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to th

Baseline mean val WMAE: 4052.72  (per-split: [np.float64(6987.3), np.float64(2610.8), np.float64(2560.1)], epochs: [14, 9, 9])


<a id='4'></a>
## 4. Model Architecture

`TemporalFusionTransformer.from_dataset(training_ds, ...)` builds the full
architecture directly from a `TimeSeriesDataSet`'s covariate roles:

- **Variable selection networks** (one for static inputs, one for the
  encoder's time-varying inputs, one for the decoder's) — gated networks
  that learn a per-variable weight at every timestep, the mechanism behind
  the "encoder/decoder/static variable importance" plots in Step 7. This is
  what makes feeding TFT the full `features.csv`/`stores.csv` covariate set
  reasonable where it wasn't for DLinear: TFT can learn to ignore what
  doesn't help, DLinear couldn't.
- **LSTM encoder-decoder** over the selected variables, giving the model a
  sequential notion of recent momentum on top of attention's longer-range
  view.
- **Interpretable multi-head attention** over the full encoder+decoder
  sequence — which past (and, for the decoder, other future) timesteps the
  model leans on for a given prediction. Source of the attention plot.
- **Gated residual networks** throughout, letting the model skip a
  transformation entirely (via a learned gate) when it isn't useful for a
  given input/context — part of why TFT tends to be more robust to
  including "maybe useful" covariates like the markdowns than a plain MLP
  would be.
- **Quantile output head** (`QuantileLoss`, default quantiles
  `[0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98]`) instead of a single point
  estimate — the source of Step 7's prediction-interval plot. The 0.5
  quantile is used as the point forecast for WMAE throughout, so results
  stay comparable to the other three notebooks.

`hidden_size` (width of the internal representations) and
`attention_head_size` (number of attention heads) are the two
capacity-controlling hyperparameters tuned in Step 5 — analogous in spirit
to DLinear's `kernel_size`, just with much more headroom given TFT's actual
capacity.

<a id='5'></a>
## 5. Hyperparameter Tuning

**Learning rate found once via Lightning's LR finder**, not grid-searched —
standard practice for this architecture (a cheap, single sweep rather than
multiplying an already-expensive grid by another dimension), run on the
largest split (`train_end=78`, the most representative amount of training
data). `hidden_size`/`attention_head_size` are still grid-searched — they
change real model capacity in ways an LR finder can't substitute for:

| Param | Values |
|---|---|
| `hidden_size` | 16, 32 |
| `attention_head_size` | 1, 4 |

4 combos, each trained across all 3 splits (12 fits total — small
deliberately, given each TFT fit costs far more than a DLinear fit),
ranked by mean WMAE across splits. Same robust-epoch-count idea as
DLinear's fix: the winning combo's 3 per-split epoch counts get **median**-
aggregated for Step 7/8's fixed training length, rather than trusting a
single run's number.

In [9]:
lr_split = cv_splits[-1]  # train_end=78, the most training data
lr_train_dl = lr_split['training_ds'].to_dataloader(train=True, batch_size=128, num_workers=0)
lr_val_dl = lr_split['val_ds'].to_dataloader(train=False, batch_size=128, num_workers=0)

lr_finder_model = TemporalFusionTransformer.from_dataset(
    lr_split['training_ds'], learning_rate=0.03, hidden_size=16, attention_head_size=1,
    dropout=0.1, hidden_continuous_size=8, loss=QuantileLoss(), optimizer='adam',
)
lr_trainer = pl.Trainer(accelerator=ACCELERATOR, enable_progress_bar=False, enable_model_summary=False, logger=False)
tuner = Tuner(lr_trainer)
lr_result = tuner.lr_find(
    lr_finder_model, train_dataloaders=lr_train_dl, val_dataloaders=lr_val_dl, min_lr=1e-4, max_lr=1.0, num_training=50,
)
LEARNING_RATE = lr_result.suggestion()
print(f'LR finder suggestion: {LEARNING_RATE:.5f}')

fig = lr_result.plot(suggest=True)
fig.savefig('plots/lr_finder_tft.png', dpi=150)
plt.show()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to th

Finding best initial lr:   0%|          | 0/50 [00:00<?, ?it/s]

INFO: `Trainer.fit` stopped: `max_steps=50` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=50` reached.
INFO: Restoring states from the checkpoint path at /content/ml-final-project/.lr_find_9d84bcb3-660c-4b7a-9b25-1e97d03a46d0.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Restoring states from the checkpoint path at /content/ml-final-project/.lr_find_9d84bcb3-660c-4b7a-9b25-1e97d03a46d0.ckpt


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL pytorch_forecasting.data.encoders.GroupNormalizer was not an allowed global by default. Please use `torch.serialization.add_safe_globals([pytorch_forecasting.data.encoders.GroupNormalizer])` or the `torch.serialization.safe_globals([pytorch_forecasting.data.encoders.GroupNormalizer])` context manager to allowlist this global if you trust this clas

In [ ]:
param_grid = {
    'hidden_size': [16, 32],
    'attention_head_size': [1, 4],
}
param_names = list(param_grid.keys())
param_combos = list(itertools.product(*param_grid.values()))
print(f'{len(param_combos)} hyperparameter combinations x {len(cv_splits)} splits = {len(param_combos) * len(cv_splits)} fits, learning_rate={LEARNING_RATE:.5f} fixed')

In [ ]:
grid_results = []
for combo_idx, combo_values in enumerate(param_combos, start=1):
    params = dict(zip(param_names, combo_values))
    mean_wmae, split_wmaes, split_epochs = train_and_score_across_splits(
        cv_splits, **params, learning_rate=LEARNING_RATE, max_epochs=MAX_EPOCHS, patience=PATIENCE,
    )
    grid_results.append({**params, 'mean_val_wmae': mean_wmae, 'split_wmaes': split_wmaes, 'split_epochs': split_epochs})
    print(f'[{combo_idx}/{len(param_combos)}] {params} -> mean_val_wmae={mean_wmae:.2f}  '
          f'(per-split: {[round(w, 1) for w in split_wmaes]}, epochs: {split_epochs})')

grid_results_df = pd.DataFrame(grid_results).sort_values('mean_val_wmae').reset_index(drop=True)
grid_results_df[['hidden_size', 'attention_head_size', 'mean_val_wmae', 'split_wmaes', 'split_epochs']]

In [ ]:
best_params = {k: int(grid_results_df.loc[0, k]) for k in param_names}
best_val_wmae = grid_results_df.loc[0, 'mean_val_wmae']
best_split_epochs = grid_results_df.loc[0, 'split_epochs']
median_best_epoch = int(round(np.median(best_split_epochs)))

print('Best params:', best_params, '(learning_rate:', LEARNING_RATE, ')')
print(f'Best mean val WMAE: {best_val_wmae:.2f} (vs baseline: {baseline_val_wmae:.2f})')
print(f'Best epochs per split: {best_split_epochs} -> median {median_best_epoch}')

<a id='6'></a>
## 6. MLflow Logging (DagsHub-hosted)

Same DagsHub-hosted MLflow server as the other three notebooks, separate
experiment `TFT_Training`. Five runs, same structure as DLinear's:

1. `TFT_Cleaning` — data shape and the local train/test split (Step 2)
2. `TFT_Windowing` — sequence-construction config, per-split dataset sizes, and the LR-finder result (Step 3/5)
3. `TFT_CV_Tuning` — the 12-fit grid search (Step 5), batched summary
4. `TFT_CV` — genuinely incremental: retrains the winning config across all 3 splits once more, logging each split's `val_loss` after every epoch as it happens
5. `TFT_Final_Fit` — the full pipeline (Step 8)

Plot artifacts are skipped (params/metrics only), same choice as the other
notebooks.

In [ ]:
import dagshub

dagshub.init(repo_owner='tgela23', repo_name='ml-final-project', mlflow=True)

import mlflow
mlflow.set_experiment('TFT_Training')
print('tracking uri:', mlflow.get_tracking_uri())

**Run 1 — `TFT_Cleaning`**

In [ ]:
with mlflow.start_run(run_name='TFT_Cleaning'):
    mlflow.log_param('train_csv_shape', str(train.shape))
    mlflow.log_param('test_csv_shape', str(test.shape))
    mlflow.log_param('train_date_range', f'{train.Date.min().date()} -> {train.Date.max().date()}')

    mlflow.log_param('local_test_holdout_weeks', 52)
    mlflow.log_param('local_train_date_range', f'{local_train_raw.Date.min().date()} -> {local_train_raw.Date.max().date()}')
    mlflow.log_param('local_test_date_range', f'{local_test_raw.Date.min().date()} -> {local_test_raw.Date.max().date()}')

    mlflow.log_metric('n_train_rows', len(train))
    mlflow.log_metric('n_local_train_rows', len(local_train_raw))
    mlflow.log_metric('n_local_test_rows', len(local_test_raw))
    mlflow.log_metric('n_stores', train['Store'].nunique())
    mlflow.log_metric('n_depts', train['Dept'].nunique())
    mlflow.log_metric('n_series', train[['Store', 'Dept']].drop_duplicates().shape[0])

print('TFT_Cleaning run logged.')

**Run 2 — `TFT_Windowing`**

In [ ]:
with mlflow.start_run(run_name='TFT_Windowing'):
    mlflow.log_param('lookback', LOOKBACK)
    mlflow.log_param('horizon_cv', HORIZON_CV)
    mlflow.log_param('horizon_final', HORIZON_FINAL)
    mlflow.log_param('cv_split_train_weeks', str(CV_SPLIT_TRAIN_WEEKS))
    mlflow.log_param('lr_finder_suggestion', LEARNING_RATE)
    mlflow.log_param('static_covariates', 'Store, Dept, Type, Size')
    mlflow.log_param('known_covariates', 'IsHoliday, Month, WeekOfYear, Temperature, Fuel_Price, CPI, Unemployment, MarkDown1-5')

    for i, s in enumerate(cv_splits):
        mlflow.log_metric(f'split{i}_train_end', s['train_end'])
        mlflow.log_metric(f'split{i}_n_training_samples', len(s['training_ds']))
    mlflow.log_metric('baseline_val_wmae', baseline_val_wmae)

print('TFT_Windowing run logged.')

**Run 3 — `TFT_CV_Tuning`**: batched summary of the 12-fit grid search.

In [ ]:
with mlflow.start_run(run_name='TFT_CV_Tuning'):
    mlflow.log_param('param_grid', str(param_grid))
    mlflow.log_param('learning_rate', LEARNING_RATE)

    for i, row in grid_results_df.iterrows():
        mlflow.log_metric('combo_mean_val_wmae', row['mean_val_wmae'], step=i)

    mlflow.log_params({f'best_{k}': v for k, v in best_params.items()})
    mlflow.log_metric('best_mean_val_wmae', best_val_wmae)
    mlflow.log_metric('median_best_epoch', median_best_epoch)

print('TFT_CV_Tuning run logged.')

**Run 4 — `TFT_CV`**: genuinely incremental companion to `TFT_CV_Tuning`
— retrains the winning hyperparameters across all 3 splits from scratch and
logs each split's `val_loss` after every epoch inside the open run, as it
happens.

In [ ]:
with mlflow.start_run(run_name='TFT_CV'):
    mlflow.log_params(best_params)
    mlflow.log_param('learning_rate', LEARNING_RATE)
    mlflow.log_param('max_epochs', MAX_EPOCHS)
    mlflow.log_param('patience', PATIENCE)

    tuned_val_wmae, tuned_split_wmaes, tuned_split_epochs = train_and_score_across_splits(
        cv_splits, **best_params, learning_rate=LEARNING_RATE, max_epochs=MAX_EPOCHS, patience=PATIENCE,
        log_mlflow=True,
    )

    mlflow.log_metric('final_mean_val_wmae', tuned_val_wmae)
    for i, e in enumerate(tuned_split_epochs):
        mlflow.log_metric(f'final_split{i}_best_epoch', e)

print(f'TFT_CV run logged. mean val WMAE = {tuned_val_wmae:.2f} '
      f'(per-split: {[round(w, 1) for w in tuned_split_wmaes]}, epochs: {tuned_split_epochs})')

<a id='7'></a>
## 7. Plots

`N_EPOCHS_FINAL` is `median(tuned_split_epochs) + 1` from Run 4 — same
robust-epoch convention as DLinear. Refit once at the tuned hyperparameters
on all 91 weeks of `local_train_raw`, this time with `max_prediction_length
= HORIZON_FINAL = 52` — a **direct** 52-week forecast, no recursive
rollout. This needs a moment of care to stay leak-free: the *training*
`TimeSeriesDataSet` is built from `local_train_raw` alone (91 weeks), so
every training window's encoder+decoder stays strictly inside those 91
weeks (verified: with `encoder<=52, decoder<=52`, the training cutoff is
`90 - 52 = 38`, so no training window ever reaches past `time_idx 90` —
`local_test_raw` never enters the training data at all). *Prediction* then
uses `TimeSeriesDataSet.from_dataset(training_ds, holdout_panel,
predict=True, ...)` on the combined `local_train_raw + local_test_raw`
panel, which correctly builds one window per series using the last 52
weeks of `local_train_raw` as encoder context and the following 52 weeks
(exactly `local_test_raw`) as the target — genuinely held-out, since
`from_dataset(..., predict=True)` only encodes data for inference, it
performs no training. All plots saved to `plots/` with a `_tft` suffix.

In [ ]:
N_EPOCHS_FINAL = median_best_epoch + 1

eval_panel = build_tft_panel(local_train_raw, features, stores)
eval_training_ds = make_training_dataset(eval_panel, LOOKBACK, HORIZON_FINAL)
eval_train_dl = eval_training_ds.to_dataloader(train=True, batch_size=128, num_workers=0)

print(f'Training final holdout-eval model for {N_EPOCHS_FINAL} fixed epochs '
      f'(median of {tuned_split_epochs} + 1) on {len(eval_training_ds)} windows...')
holdout_model = TemporalFusionTransformer.from_dataset(
    eval_training_ds, learning_rate=LEARNING_RATE, **best_params, dropout=0.1,
    hidden_continuous_size=max(best_params['hidden_size'] // 2, 4), loss=QuantileLoss(), optimizer='adam',
)
holdout_trainer = pl.Trainer(
    max_epochs=N_EPOCHS_FINAL, accelerator=ACCELERATOR, enable_progress_bar=False, enable_model_summary=False,
    logger=False, enable_checkpointing=False,
)
holdout_trainer.fit(holdout_model, train_dataloaders=eval_train_dl)

holdout_panel = build_tft_panel(pd.concat([local_train_raw, local_test_raw], ignore_index=True), features, stores)
holdout_pred_ds = TimeSeriesDataSet.from_dataset(eval_training_ds, holdout_panel, predict=True, stop_randomization=True)

In [ ]:
holdout_raw = holdout_model.predict(holdout_pred_ds, mode='raw', return_index=True, return_x=True, batch_size=256, num_workers=0)
holdout_median = np.clip(holdout_model.predict(holdout_pred_ds, mode='prediction', batch_size=256, num_workers=0).cpu().numpy(), 0, None)
holdout_index = holdout_raw.index

pred_rows = []
for row_i in range(holdout_median.shape[0]):
    store, dept = str(int(holdout_index.iloc[row_i]['Store'])), str(int(holdout_index.iloc[row_i]['Dept']))
    start_idx = int(holdout_index.iloc[row_i]['time_idx'])
    sub = holdout_panel[(holdout_panel['Store'].astype(str) == store) & (holdout_panel['Dept'].astype(str) == dept) &
                         (holdout_panel['time_idx'] >= start_idx) & (holdout_panel['time_idx'] < start_idx + HORIZON_FINAL)]
    sub = sub.sort_values('time_idx')
    for step, (_, r) in enumerate(sub.iterrows()):
        pred_rows.append((int(store), int(dept), r['Date'], r['Weekly_Sales'], holdout_median[row_i, step], bool(r['IsHoliday'])))

pred_df = pd.DataFrame(pred_rows, columns=['Store', 'Dept', 'Date', 'Actual', 'Predicted', 'IsHoliday'])
pred_df['Residual'] = pred_df['Actual'] - pred_df['Predicted']

holdout_wmae = wmae(pred_df['Actual'], pred_df['Predicted'], pred_df['IsHoliday'])
print(f'Local-test holdout WMAE (tuned model, direct 52-week forecast, no recursion): {holdout_wmae:.2f}')

**Plot 1 — Actual vs. predicted over time, with quantile uncertainty
band**, using `pytorch_forecasting`'s native `plot_prediction` — the same 3
sample Store/Dept series used in every other notebook, now with TFT's
calibrated prediction interval shown directly (something none of the other
three models provide).

In [ ]:
sample_combos = [(1, 1), (1, 72), (20, 1)]
sample_series_keys = list(zip(holdout_index['Store'].astype(int), holdout_index['Dept'].astype(int)))

for store, dept in sample_combos:
    if (store, dept) not in sample_series_keys:
        print(f'Store {store}, Dept {dept} not in the holdout prediction set, skipping.')
        continue
    idx = sample_series_keys.index((store, dept))
    fig = holdout_model.plot_prediction(holdout_raw.x, holdout_raw.output, idx=idx, add_loss_to_title=False)
    fig.suptitle(f'TFT: actual vs. predicted (with quantile band) — Store {store}, Dept {dept}')
    fig.savefig(f'plots/actual_vs_predicted_timeseries_tft_store{store}_dept{dept}.png', dpi=150, bbox_inches='tight')
    plt.show()

**Plots 2-5 — Native TFT interpretation**: encoder variable importance,
decoder variable importance, static variable importance, and temporal
attention — all from `pytorch_forecasting`'s built-in
`interpret_output`/`plot_interpretation`, genuine model-internal quantities
(not a proxy like DLinear's linear-weight-magnitude plot).

In [ ]:
interpretation = holdout_model.interpret_output(holdout_raw.output, reduction='sum')
interp_figs = holdout_model.plot_interpretation(interpretation)
for name, fig in interp_figs.items():
    fig.savefig(f'plots/interpretation_{name}_tft.png', dpi=150, bbox_inches='tight')
    plt.show()

**Plot 6 — WMAE by stage**: baseline (untuned, mean across splits) vs.
tuned (mean across splits) vs. final holdout — same comparison as the other
two DL notebooks.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
stages = ['Baseline\n(mean, 3 splits)', 'Tuned\n(mean, 3 splits)', 'Final\n(holdout)']
values = [baseline_val_wmae, tuned_val_wmae, holdout_wmae]
colors = ['#B0B0B0', '#4C72B0', '#55A868']
ax.bar(stages, values, color=colors)
for i, v in enumerate(values):
    ax.text(i, v + max(values) * 0.01, f'{v:.0f}', ha='center')
ax.set_ylabel('WMAE')
ax.set_title('WMAE by stage — baseline vs. tuned vs. final holdout (TFT)')
plt.tight_layout()
plt.savefig('plots/wmae_by_fold_tft.png', dpi=150)
plt.show()

**Plot 7 — Residual distribution** on the local-test holdout.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(pred_df['Residual'], bins=60, color='#4C72B0', alpha=0.8)
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('Residual (Actual - Predicted)')
ax.set_title('Residual distribution — local-test holdout (TFT)')
plt.tight_layout()
plt.savefig('plots/residual_distribution_tft.png', dpi=150)
plt.show()

**Plot 8 — Actual vs. predicted, holiday vs. non-holiday weeks**

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
for is_holiday, color, label in [(False, '#4C72B0', 'Non-holiday'), (True, '#C44E52', 'Holiday')]:
    sub = pred_df[pred_df['IsHoliday'] == is_holiday]
    ax.scatter(sub['Actual'], sub['Predicted'], alpha=0.3, s=10, color=color, label=label)
lims = [0, pred_df[['Actual', 'Predicted']].to_numpy().max() * 1.05]
ax.plot(lims, lims, color='black', linewidth=1, linestyle='--')
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
ax.set_title('Actual vs. Predicted — Holiday vs. Non-holiday weeks (TFT)')
ax.legend()
plt.tight_layout()
plt.savefig('plots/actual_vs_predicted_holiday_tft.png', dpi=150)
plt.show()

<a id='8'></a>
## 8. Full Pipeline

`utils.tft.TFTForecastPipeline` is the TFT analogue of
`DLinearForecastPipeline`/the tree notebooks' `sklearn.pipeline.Pipeline`:
`fit()` stores the raw training history and the fitted `TimeSeriesDataSet`
encoding; `predict()` takes bare `Store/Dept/Date/IsHoliday` rows (e.g.
`test.csv` exactly as downloaded), merges in `features.csv`/`stores.csv`
the same way `build_tft_panel` does, and returns median (0.5 quantile)
`Weekly_Sales` predictions — no manual feature work by the caller. Series
present in a prediction request but absent from the fitted history get NaN
(documented limitation, not silently zero-filled), same contract as
`DLinearForecastPipeline`.

Fit on **all of `train.csv`** (143 weeks, all 3,331 series — everything
before the real Kaggle `test.csv`, now that tuning/plots are done and
there's no more need to hold anything out), `max_prediction_length =
HORIZON_FINAL = 52` — covering Kaggle's real 39-week `test.csv` as a strict
subset — for the same `N_EPOCHS_FINAL` fixed epochs as Step 7's holdout
model.

In [ ]:
full_pipeline = TFTForecastPipeline(
    model=None, max_encoder_length=LOOKBACK, max_prediction_length=HORIZON_FINAL,
    features=features, stores=stores, device=ACCELERATOR,
)
full_pipeline.fit(train)  # builds history_panel_ / training_dataset_; model attached below

full_train_dl = full_pipeline.training_dataset_.to_dataloader(train=True, batch_size=128, num_workers=0)
print(f'{len(full_pipeline.training_dataset_)} training windows across all of train.csv')

print(f'Training final model for {N_EPOCHS_FINAL} fixed epochs...')
final_model = TemporalFusionTransformer.from_dataset(
    full_pipeline.training_dataset_, learning_rate=LEARNING_RATE, **best_params, dropout=0.1,
    hidden_continuous_size=max(best_params['hidden_size'] // 2, 4), loss=QuantileLoss(), optimizer='adam',
)
final_trainer = pl.Trainer(
    max_epochs=N_EPOCHS_FINAL, accelerator=ACCELERATOR, enable_progress_bar=False, enable_model_summary=False,
    logger=False, enable_checkpointing=False,
)
final_trainer.fit(final_model, train_dataloaders=full_train_dl)

full_pipeline.model = final_model
print('TFTForecastPipeline fit on all of train.csv.')

**Confirm it truly takes raw input**: predict on bare rows straight from
`test.csv` — unmerged, no `features.csv`/`stores.csv` merge or history
computed by the caller.

In [ ]:
raw_sample = test.head(5)
print('Raw input columns (exactly test.csv, nothing pre-computed):', raw_sample.columns.tolist())

raw_preds = full_pipeline.predict(raw_sample[['Store', 'Dept', 'Date', 'IsHoliday']])
print()
print('Predictions:', raw_preds)

**Save to MLflow (DagsHub model registry)** inside `TFT_Final_Fit`.
`mlflow.pyfunc` (not a raw Lightning checkpoint) for the registry entry,
same reasoning as the tree notebooks' `serialization_format='cloudpickle'`
and DLinear's `DLinearPyfuncWrapper`: the saved artifact needs to be the
whole raw-input-to-prediction path, not just the bare model.

In [ ]:
from mlflow.models import infer_signature


class TFTPyfuncWrapper(mlflow.pyfunc.PythonModel):
    def __init__(self, pipeline):
        self.pipeline = pipeline

    def predict(self, context, model_input, params=None):
        return self.pipeline.predict(model_input)


signature = infer_signature(raw_sample[['Store', 'Dept', 'Date', 'IsHoliday']], raw_preds)

with mlflow.start_run(run_name='TFT_Final_Fit'):
    mlflow.log_params(best_params)
    mlflow.log_param('learning_rate', LEARNING_RATE)
    mlflow.log_param('lookback', LOOKBACK)
    mlflow.log_param('horizon_final', HORIZON_FINAL)
    mlflow.log_param('n_epochs_final', N_EPOCHS_FINAL)
    mlflow.log_metric('local_test_holdout_wmae', holdout_wmae)
    mlflow.log_metric('n_series_trained', train[['Store', 'Dept']].drop_duplicates().shape[0])

    mlflow.pyfunc.log_model(
        artifact_path='model',
        python_model=TFTPyfuncWrapper(full_pipeline),
        signature=signature,
        input_example=raw_sample[['Store', 'Dept', 'Date', 'IsHoliday']],
    )

print('TFT_Final_Fit run logged, pipeline saved to MLflow model registry.')

**Also save locally** under `models/` — via Lightning's own checkpoint
mechanism (`trainer.save_checkpoint`), the more robust choice for a
Lightning module than generic pickling, plus the raw history/stats
`TFTForecastPipeline` needs, saved alongside with `joblib`.

In [ ]:
import os
import joblib

os.makedirs('models', exist_ok=True)
final_trainer.save_checkpoint('models/tft_model.ckpt')
joblib.dump({
    'history_panel_': full_pipeline.history_panel_,
    'training_dataset_': full_pipeline.training_dataset_,
    '_raw_train_df': full_pipeline._raw_train_df,
    '_series_': full_pipeline._series_,
    'last_date_': full_pipeline.last_date_,
    'max_encoder_length': LOOKBACK,
    'max_prediction_length': HORIZON_FINAL,
}, 'models/tft_pipeline_state.joblib')
print('Saved to models/tft_model.ckpt and models/tft_pipeline_state.joblib')

In [ ]:
reloaded_model = TemporalFusionTransformer.load_from_checkpoint('models/tft_model.ckpt')
state = joblib.load('models/tft_pipeline_state.joblib')

reloaded_pipeline = TFTForecastPipeline(
    model=reloaded_model, max_encoder_length=state['max_encoder_length'],
    max_prediction_length=state['max_prediction_length'], features=features, stores=stores, device=ACCELERATOR,
)
reloaded_pipeline.history_panel_ = state['history_panel_']
reloaded_pipeline.training_dataset_ = state['training_dataset_']
reloaded_pipeline._raw_train_df = state['_raw_train_df']
reloaded_pipeline._series_ = state['_series_']
reloaded_pipeline.last_date_ = state['last_date_']

reloaded_preds = reloaded_pipeline.predict(raw_sample[['Store', 'Dept', 'Date', 'IsHoliday']])
print('Reloaded-pipeline predictions match:', np.allclose(reloaded_preds, raw_preds, rtol=1e-4))